In [ ]:
import requests
import json
import time
from datetime import datetime

TOPIC = "universal healthcare"
HEADERS = {"User-Agent": "tension-analyzer/1.0"}

subreddits = ["politics", "Conservative", "Liberal", "healthcare", "PoliticalDiscussion"]

data = []

for sub in subreddits:
    url = f"https://www.reddit.com/r/{sub}/search.json?q={TOPIC}&sort=new&limit=10&restrict_sr=1"
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        
        if response.status_code == 200:
            posts = response.json()["data"]["children"]
            for post in posts:
                p = post["data"]
                entry = {
                    "title": p["title"],
                    "body": p.get("selftext", "")[:300],
                    "subreddit": p["subreddit"],
                    "score": p["score"],
                    "num_comments": p["num_comments"],
                    "url": f"https://reddit.com{p['permalink']}",
                    "created_utc": p["created_utc"],
                }
                data.append(entry)
                print(f"[r/{entry['subreddit']}] {entry['title'][:80]}")
        else:
            print(f"r/{sub} returned status {response.status_code}")
            
    except Exception as e:
        print(f"r/{sub} failed: {e}")
    
    time.sleep(1)

print(f"\n✓ Collected {len(data)} posts")

# ── Save to JSON ───────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"reddit_{TOPIC.replace(' ', '_')}_{timestamp}.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump({
        "topic": TOPIC,
        "collected_at": timestamp,
        "total_posts": len(data),
        "posts": data
    }, f, ensure_ascii=False, indent=2)

print(f"✓ Saved to {filename}")